# Generatie van Synthetische Storingsdata (Synthetic Data Generation)

**Auteur**: `Jakob De Vreese`  
**Bachelorproef**: Overdraagbare unsupervised anomaliedetectie bij hybride HVAC-systemen  
**Academiejaar**: `2025-2026`  

## Doel

In deze notebook genereren we een kwalitatieve, **gelabelde synthetische dataset** door gecontroleerd afwijkingen (anomalieën) te injecteren in de historische tijdreeksen van gevalideerd normaalgedrag. Omdat we de *unsupervised* machine learning-modellen objectief en accuraat willen evalueren, construeren we hier een betrouwbare *ground truth*. De focus ligt op het realistisch simuleren van de 10 storingsscenario's die tijdens de brainstormsessies met de HVAC-domeinexperten zijn geïdentificeerd.

### Doel van deze notebook

- Wiskundig modelleren van de fysieke impact van de geselecteerde storingsprofielen (*fault signatures*) op de HVAC-sensordata.
- Stochastisch (willekeurig) en dynamisch injecteren van deze fouten over de beschikbare tijdlijn.
- Creëren van een representatieve validatieset met binaire labels (0 = normaal, 1 = anomalie) om de detectieprestaties (F1-scores) van de algoritmes te kunnen meten.
- Het contrast tussen het normale baseline-gedrag en de gesimuleerde afwijkingen inzichtelijk maken via visualisaties.

## Opbouw van de notebook

- **Sectie 1**: Inladen en voorbereiden van de opgeschoonde baseline-dataset (normaalgedrag)
- **Sectie 2**: Wiskundige definitie en configuratie van de 10 geselecteerde storingsprofielen
- **Sectie 3**: Ontwikkeling van het stochastische injectie-algoritme
- **Sectie 4**: Generatie van de afwijkende tijdreeksen (Data simulatie)
- **Sectie 5**: Visualisatie en validatie van de geïnjecteerde anomalieën per categorie (acuut vs. sluipend)
- **Sectie 6**: Export van de finale gelabelde dataset voor modelevaluatie

### Verwachte output van deze notebook

- Een robuuste, gelabelde test- en validatiedataset (`.csv` formaat) klaar voor gebruik in de model-evaluatiefase.
- Heldere tijdreeksvisualisaties die aantonen hoe de artificiële fouten zich manifesteren in de meetdata van de verschillende subsystemen (opwekking, ventilatie, afgifte).

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from pandas.api import types as ptypes
from typing import Tuple, List, Dict

## Sectie 1: Inladen en voorbereiden van de opgeschoonde baseline-dataset

In [10]:
# Hyperparameters
GEBOUWNAAM = 'dunant1'

In [11]:
# Inlezen dataset 
url = f'../../data/processed/{GEBOUWNAAM}.csv'
data = pd.read_csv(url, parse_dates=['timestamp'], index_col='timestamp').sort_index()

data.head()

,f_1,f_2,f_3,f_4,f_6,f_7,f_8,f_9,f_10,f_11,...,f_50,f_51,f_52,f_53,f_54,f_55,is_weekdag,is_weekend,is_werkuur,uur
timestamp,,,,,,,,,,,,,,,,,,,,,
2026-03-09 00:00:00+00:00,0.0,0.00,1.0,39.038220,30.977285,1.0,18.945000,0.000000,0.0,18.000000,...,0.0,4.223216,98.886563,0.639789,100.795459,0.0,1,0,0,0
2026-03-09 00:10:00+00:00,7.0,0.77,1.0,39.877285,31.832268,1.0,23.594538,44.640636,1.0,22.222490,...,0.0,3.851797,99.589400,0.749325,86.946641,0.0,1,0,0,0
2026-03-09 00:20:00+00:00,9.0,0.78,1.0,41.921010,34.104786,1.0,46.930084,44.640636,1.0,38.731342,...,0.0,3.903977,100.000000,0.559565,45.771835,0.0,1,0,0,0
2026-03-09 00:30:00+00:00,8.0,0.86,1.0,43.412975,35.585346,1.0,43.235270,44.640636,0.0,41.083800,...,0.0,4.244891,99.196013,0.322963,91.228401,0.0,1,0,0,0
2026-03-09 00:40:00+00:00,7.0,0.78,1.0,43.734930,35.924030,1.0,48.845810,44.640636,0.0,46.000000,...,0.0,4.448537,98.800579,0.300000,323.796930,0.0,1,0,0,0


### Splitsen van dataset in train en test

We gaan nu de dataset die we ingeladen hebben splitsen in een dataset waarmee we gaan trainen, en houden een stuk van `10%` van de data opzij om later te testen, de volledige dataset is verondersteld normaalgedrag te weerspiegelen, op die 10% data zullen we dus fouten genereren in volgende secties.

In [12]:
# Stap 1: Dataset spliten (10% test, 90% train)
print("=" * 60)
print("DATASET SPLITTING")
print("=" * 60)

# Bereken split point (10% van dataset)
split_idx = int(len(data) * 0.9)

# Neem laatste 10% aaneensluitend (meest recente data)
test_data_raw = data.iloc[split_idx:].copy()
train_data = data.iloc[:split_idx].copy()

print(f"Total dataset: {len(data):,} rijen ({len(data) / (60/10) / 24:.1f} dagen)")
print(f"Train (90%): {len(train_data):,} rijen ({len(train_data) / (60/10) / 24:.1f} dagen)")
print(f"Test (10%): {len(test_data_raw):,} rijen ({len(test_data_raw) / (60/10) / 24:.1f} dagen)")

# Sla trainingsdata op
train_output = f'../../data/processed/{GEBOUWNAAM}_train.csv'
train_data.to_csv(train_output)
print(f"\n✓ Trainingsdata opgeslagen: {train_output}")

DATASET SPLITTING
Total dataset: 5,040 rijen (35.0 dagen)
Train (90%): 4,536 rijen (31.5 dagen)
Test (10%): 504 rijen (3.5 dagen)

✓ Trainingsdata opgeslagen: ../../data/processed/dunant1_train.csv


## Sectie 2: Wiskundige foutgeneratie

De fouten die in dit script wiskundig gemodelleerd worden, zijn zorgvuldig geselecteerd op basis van *knowledge elicitation* (brainstormsessies) met de domeinexperten van de exploitatieploeg HVAC. Ze omvatten vier kwadranten:

1. **Abrupt & Grote impact:** Acuut falen warmtepomp / Vastgelopen batterijklep luchtgroep.
2. **Sluipend (Soft faults) & Grote impact:** Prestatiedegradatie (koudemiddellek) warmtepomp / Filtervervuiling / Defecte warmteterugwinning (WTW).
3. **Abrupt & Lage impact:** Defecte (bevroren) ruimtevoeler / Open raam tijdens verwarmingsregime.
4. **Sluipend (Soft faults) & Lage impact:** Pendelgedrag (start/stops) gasketel / Falende nachtverlaging / Lekkende emissieklep.

In [ ]:
class FaultSimulator:
    """
    Injecteert synthetische HVAC-fouten in een gelabelde testdataset.
    Kolommen zijn f_1 tot f_51 (6-urige voorkeursdiameter).
    VEREENVOUDIGD EN MEMORY-SAFE design.
    """
    
    def __init__(self, data: pd.DataFrame, random_seed: int = 42):
        self.data = data.copy()
        self.anomaly_labels = np.zeros(len(data), dtype=int)
        self.fault_ranges = []
        np.random.seed(random_seed)
        self.frequency_min = 10
        
        # Buffer dynamisch aanpassen op basis van dataset grootte
        dataset_days = len(data) * self.frequency_min / 60 / 24
        if dataset_days < 10:
            # Voor kleine datasets (< 10 dagen): 1 uur buffer
            self.min_gap_between_faults = 60
        elif dataset_days < 30:
            # Voor medium datasets (< 30 dagen): 2 uur buffer
            self.min_gap_between_faults = 120
        else:
            # Voor grote datasets (> 30 dagen): 6 uur buffer
            self.min_gap_between_faults = 360
        
        print(f"  Dataset grootte: {dataset_days:.1f} dagen")
        print(f"  Buffer tussen fouten: {self.min_gap_between_faults} min\n")
        
    def _find_valid_window(self, duration_minutes: int) -> Tuple[int, int]:
        """Zoek beschikbare slot zonder overlappen."""
        duration_steps = max(1, duration_minutes // self.frequency_min)
        buffer_steps = self.min_gap_between_faults // self.frequency_min
        
        for attempt in range(50):
            start = np.random.randint(0, max(1, len(self.data) - duration_steps))
            end = start + duration_steps
            
            if end >= len(self.data):
                continue
            
            conflict = False
            for fault_start, fault_end in self.fault_ranges:
                if not (end + buffer_steps < fault_start or start > fault_end + buffer_steps):
                    conflict = True
                    break
            
            if not conflict:
                return start, end
        
        return None, None
    
    def inject_fault_1_heatpump_failure(self, duration_min: int = 30):
        """Fout 1: Acuut falen warmtepomp"""
        start, end = self._find_valid_window(duration_min)
        if start is None:
            print(f"  ⚠ Fout 1: Kan geen window vinden")
            return
        
        fault_sensors = ['f_1', 'f_2', 'f_3', 'f_7']
        for col in fault_sensors:
            if col in self.data.columns:
                self.data.iloc[start:end, self.data.columns.get_loc(col)] = 0
        
        self.anomaly_labels[start:end] = 1
        self.fault_ranges.append((start, end))
        print(f"  ✓ Fout 1 (HP fail): {duration_min} min")
    
    def inject_fault_2_stuck_battery_valve(self, duration_min: int = 60):
        """Fout 2: Vastgelopen klep"""
        start, end = self._find_valid_window(duration_min)
        if start is None:
            print(f"  ⚠ Fout 2: Kan geen window vinden")
            return
        
        stuck_pos = np.random.uniform(0.8, 1.0)
        
        if 'f_29' in self.data.columns:
            self.data.iloc[start:end, self.data.columns.get_loc('f_29')] = stuck_pos
        
        if 'f_22' in self.data.columns and 'f_21' in self.data.columns:
            setpoints = self.data.iloc[start:end, self.data.columns.get_loc('f_21')].values
            self.data.iloc[start:end, self.data.columns.get_loc('f_22')] = setpoints * 0.7
        
        if 'f_30' in self.data.columns and 'f_31' in self.data.columns:
            temp_vertrek = self.data.iloc[start:end, self.data.columns.get_loc('f_30')].values
            self.data.iloc[start:end, self.data.columns.get_loc('f_31')] = temp_vertrek
        
        self.anomaly_labels[start:end] = 1
        self.fault_ranges.append((start, end))
        print(f"  ✓ Fout 2 (Stuck valve): {duration_min} min")
    
    def inject_fault_3_heatpump_degradation(self, duration_min: int = 360):
        """Fout 3: Prestatiedegradatie warmtepomp"""
        start, end = self._find_valid_window(duration_min)
        if start is None:
            print(f"  ⚠ Fout 3: Kan geen window vinden")
            return
        
        if 'f_3' in self.data.columns:
            status = self.data.iloc[start:end, self.data.columns.get_loc('f_3')].values
            self.data.iloc[start:end, self.data.columns.get_loc('f_3')] = np.where(status > 0.5, 1, status)
        
        if 'f_4' in self.data.columns and 'f_6' in self.data.columns:
            supply = self.data.iloc[start:end, self.data.columns.get_loc('f_4')].values
            return_t = self.data.iloc[start:end, self.data.columns.get_loc('f_6')].values
            new_delta = (supply - return_t) * 0.4
            self.data.iloc[start:end, self.data.columns.get_loc('f_4')] = return_t + new_delta
        
        self.anomaly_labels[start:end] = 1
        self.fault_ranges.append((start, end))
        print(f"  ✓ Fout 3 (Degradatie): {duration_min} min")
    
    def inject_fault_4_filter_clogging(self, duration_min: int = 240):
        """Fout 4: Filtervervuiling"""
        start, end = self._find_valid_window(duration_min)
        if start is None:
            print(f"  ⚠ Fout 4: Kan geen window vinden")
            return
        
        if 'f_27' in self.data.columns:
            current = self.data.iloc[start:end, self.data.columns.get_loc('f_27')].values
            steps = np.linspace(0, 0.2, len(current))
            self.data.iloc[start:end, self.data.columns.get_loc('f_27')] = np.clip(current + steps, 0, 1)
        
        self.anomaly_labels[start:end] = 1
        self.fault_ranges.append((start, end))
        print(f"  ✓ Fout 4 (Filter clogging): {duration_min} min")
    
    def inject_fault_6_sensor_failure(self, duration_min: int = 120):
        """Fout 6: Sensordefect"""
        start, end = self._find_valid_window(duration_min)
        if start is None:
            print(f"  ⚠ Fout 6: Kan geen window vinden")
            return
        
        sensor_col = np.random.choice(['f_44', 'f_48'])
        
        if sensor_col in self.data.columns:
            frozen_val = self.data.iloc[start, self.data.columns.get_loc(sensor_col)]
            self.data.iloc[start:end, self.data.columns.get_loc(sensor_col)] = frozen_val
        
        self.anomaly_labels[start:end] = 1
        self.fault_ranges.append((start, end))
        print(f"  ✓ Fout 6 (Sensor freeze - {sensor_col}): {duration_min} min")
    
    def inject_fault_7_open_window(self, duration_min: int = 60):
        """Fout 7: Open raam"""
        start, end = self._find_valid_window(duration_min)
        if start is None:
            print(f"  ⚠ Fout 7: Kan geen window vinden")
            return
        
        zone_choice = np.random.choice([1, 2])
        temp_col = f'f_{44 + (zone_choice-1)*4}'
        co2_col = f'f_{45 + (zone_choice-1)*4}'
        
        if temp_col in self.data.columns:
            baseline = self.data.iloc[start, self.data.columns.get_loc(temp_col)]
            decline = np.linspace(0, -4, len(range(start, end)))
            self.data.iloc[start:end, self.data.columns.get_loc(temp_col)] = baseline + decline
        
        if co2_col in self.data.columns:
            self.data.iloc[start:end, self.data.columns.get_loc(co2_col)] = 400
        
        self.anomaly_labels[start:end] = 1
        self.fault_ranges.append((start, end))
        print(f"  ✓ Fout 7 (Open window - zone {zone_choice}): {duration_min} min")
    
    def inject_fault_8_boiler_cycling(self, duration_min: int = 45):
        """Fout 8: Gasketel pendelgedrag"""
        start, end = self._find_valid_window(duration_min)
        if start is None:
            print(f"  ⚠ Fout 8: Kan geen window vinden")
            return
        
        if 'f_10' in self.data.columns:
            cycle = np.tile([1, 1, 1, 0, 0, 0], (len(range(start, end)) // 6) + 1)
            self.data.iloc[start:end, self.data.columns.get_loc('f_10')] = cycle[:len(range(start, end))]
        
        self.anomaly_labels[start:end] = 1
        self.fault_ranges.append((start, end))
        print(f"  ✓ Fout 8 (Boiler cycling): {duration_min} min")
    
    def inject_fault_9_no_night_setback(self, duration_min: int = 480):
        """Fout 9: Geen nachtverlaging"""
        start, end = self._find_valid_window(duration_min)
        if start is None:
            print(f"  ⚠ Fout 9: Kan geen window vinden")
            return
        
        for col in ['f_38', 'f_39']:
            if col in self.data.columns:
                self.data.iloc[start:end, self.data.columns.get_loc(col)] = 1.0
        
        self.anomaly_labels[start:end] = 1
        self.fault_ranges.append((start, end))
        print(f"  ✓ Fout 9 (No night setback): {duration_min} min")
    
    def inject_random_faults(self, num_faults: int = None) -> Dict:
        """Injecteer random aantal fouten (2-4)"""
        if num_faults is None:
            num_faults = np.random.randint(2, 5)
        
        available_faults = [
            ('Fout 1: HP Fail', self.inject_fault_1_heatpump_failure),
            ('Fout 2: Stuck Valve', self.inject_fault_2_stuck_battery_valve),
            ('Fout 3: Degradatie', self.inject_fault_3_heatpump_degradation),
            ('Fout 4: Filter', self.inject_fault_4_filter_clogging),
            ('Fout 6: Sensor', self.inject_fault_6_sensor_failure),
            ('Fout 7: Open Window', self.inject_fault_7_open_window),
            ('Fout 8: Boiler Cycle', self.inject_fault_8_boiler_cycling),
            ('Fout 9: No Night', self.inject_fault_9_no_night_setback),
        ]
        
        # Select random
        indices = np.random.choice(len(available_faults), size=min(num_faults, len(available_faults)), replace=False)
        selected = [available_faults[i] for i in indices]
        
        print(f"\n{'='*60}")
        print(f"INJECTIE VAN {len(selected)} SYNTHETISCHE FOUTEN")
        print(f"{'='*60}\n")
        
        for name, method in selected:
            try:
                method()
            except Exception as e:
                print(f"  ✗ {name} error: {e}")
        
        return {name: method for name, method in selected}
    
    def get_result(self) -> Tuple[pd.DataFrame, np.ndarray]:
        """Retourneer resultaten"""
        return self.data, self.anomaly_labels
    
    def summary(self):
        """Print samenvatting"""
        num_anomalies = np.sum(self.anomaly_labels)
        pct_anomalies = (num_anomalies / len(self.anomaly_labels)) * 100 if len(self.anomaly_labels) > 0 else 0
        
        print(f"\n{'='*60}")
        print(f"SAMENVATTING SYNTHETISCHE DATASET")
        print(f"{'='*60}")
        print(f"Totaal rijen: {len(self.data):,}")
        print(f"Anomalie rijen: {num_anomalies:,} ({pct_anomalies:.1f}%)")
        print(f"Normale rijen: {len(self.data) - num_anomalies:,} ({100-pct_anomalies:.1f}%)")
        print(f"Aantal fault ranges: {len(self.fault_ranges)}")
        print(f"{'='*60}\n")


In [15]:
# Maak simulator aan
print("\n" + "="*60)
print("FAULT SIMULATION START")
print("="*60)

simulator = FaultSimulator(test_data_raw, random_seed=42)
fault_info = simulator.inject_random_faults(num_faults=5)  # of None voor random 2-4

# Resultaat ophalen
test_data, labels = simulator.get_result()

# Samenvatting
simulator.summary()

# Opslaan
test_output = f'../../data/processed/{GEBOUWNAAM}_test.csv'
labels_output = f'../../data/processed/{GEBOUWNAAM}_test_labels.npy'

test_data.to_csv(test_output)
np.save(labels_output, labels)

print(f"✓ Test data opgeslagen: {test_output}")
print(f"✓ Labels opgeslagen: {labels_output}")


FAULT SIMULATION START
  Dataset grootte: 3.5 dagen
  Buffer tussen fouten: 60 min


INJECTIE VAN 5 SYNTHETISCHE FOUTEN

  ✓ Fout 2 (Stuck valve): 60 min
  ✓ Fout 7 (Open window - zone 1): 60 min
  ✓ Fout 1 (HP fail): 30 min
  ✓ Fout 9 (No night setback): 480 min
  ✓ Fout 3 (Degradatie): 360 min

SAMENVATTING SYNTHETISCHE DATASET
Totaal rijen: 504
Anomalie rijen: 99 (19.6%)
Normale rijen: 405 (80.4%)
Aantal fault ranges: 5

✓ Test data opgeslagen: ../../data/processed/dunant1_test.csv
✓ Labels opgeslagen: ../../data/processed/dunant1_test_labels.npy
